In [1]:
import wandb
import pandas as pd
import numpy as np

import json

api = wandb.Api()

from helpers import *

## IAUC %

In [2]:
iauc_p_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.percentage": {"$ne": None}},
            {"tags": {"$nin": ["Reverse"]}},
            {"$or": [
                {'config.dataset.datasets.val_pascal5i_N1K1.name': "pascal"},
                {'config.dataset.datasets.val_coco20i_N1K1.name': "coco"},
            ]}
        ]
    }
)

In [3]:
iauc_p_runs_df = get_runs_df(iauc_p_runs)

In [4]:
res_df = iauc_p_runs_df.copy()

dataset_names = ["val_pascal5i_N1K1", "val_coco20i_N1K1"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1)

iauc_cols = [f"{name}_iauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."
res_df = res_df.rename(columns=renamings_dict)


res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["iAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["Loss"] = res_df["Loss"].fillna(False)

cols = ["Model", "Explanation Method", "Dataset", "Percentage", "Loss"]
res_df = res_df[cols + ["iAUC"]]

res_df = res_df.pivot_table(
    index=['Model', 'Explanation Method'],
    columns=['Dataset', 'Percentage', "Loss"],
    values='iAUC'
).sort_index(axis=1)

perc_map = {
    False: "IAUC@",
    True: "mIoULoss@"
}

# Flatten the MultiIndex columns
# Collapse last two levels into one
res_df.columns = pd.MultiIndex.from_tuples([
    (lvl0, f"{perc_map[lvl2]}{lvl1}") for lvl0, lvl1, lvl2 in res_df.columns
])


# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Model', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(model_order) if x.name == 'Model' else x.map(method_order)
)

iauc_p_df = (res_df*100).round(2)

iauc_p_df

/tmp/ipykernel_1137718/1354437377.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  res_df["Loss"] = res_df["Loss"].fillna(False)


COCO 20^i               Pascal 5^i  \
                             mIoULoss@0.01 mIoULoss@0.05  IAUC@0.01   
Model  Explanation Method                                             
DCAMA  Random                        33.79         29.78      44.44   
       Gaussian Noise Mask           27.07         12.96      46.95   
       Saliency                      33.93         24.13      39.69   
       Integrated Gradients          32.19         22.61      40.93   
       Guided IG                     25.96         19.49      42.66   
       Blur IG                       33.89         23.29      40.38   
       XRAI                          29.65         14.50      45.90   
       Deep Lift                     32.60         24.85      41.01   
       LIME                          27.27         13.44      49.56   
       Unmasked AffEx (ours)         19.84         10.08      55.76   
       Masked AffEx (ours)           17.63          8.11        NaN   
       Signed AffEx (ours)           18.71          9.82        NaN   
DMTNet Random                        23.53         26.01      55.00   
       Gaussian Noise Mask           16.20          8.66      52.97   
       Saliency                      20.15         17.46      53.76   
       Integrated Gradients          20.53         20.81      53.45   
       Blur IG                       20.22         17.42      53.35   
       XRAI                          14.83          6.30      56.53   
       Deep Lift                     20.81         22.25      54.07   
       LIME                          14.84          8.32      58.51   
       Unmasked AffEx (ours)         10.97          4.14      53.42   
       Masked AffEx (ours)           10.09          3.77        NaN   
       Signed AffEx (ours)           11.12          3.80        NaN   

                                                          
                             mIoULoss@0.01 mIoULoss@0.05  
Model  Explanation Method                                 
DCAMA  Random                        48.72         34.64  
       Gaussian Noise Mask           33.60         16.01  
       Saliency                      50.06         32.80  
       Integrated Gradients          44.41         27.53  
       Guided IG                     41.33         29.58  
       Blur IG                       47.96         28.61  
       XRAI                          37.49         13.25  
       Deep Lift                     46.76         28.93  
       LIME                          40.03         13.97  
       Unmasked AffEx (ours)         22.85          7.90  
       Masked AffEx (ours)           19.33          7.32  
       Signed AffEx (ours)           21.80          9.72  
DMTNet Random                        35.79         41.19  
       Gaussian Noise Mask           31.02         23.73  
       Saliency                      33.30         35.59  
       Integrated Gradients          32.03         37.64  
       Blur IG                       31.56         35.04  
       XRAI                          20.56         10.80  
       Deep Lift                     32.10         39.42  
       LIME                          22.95         10.40  
       Unmasked AffEx (ours)         16.35          7.85  
       Masked AffEx (ours)           16.49          7.88  
       Signed AffEx (ours)           16.36          9.45

In [5]:
from tabulate import tabulate

# turn the two level index into two columns
final_to_latex = iauc_p_df.reset_index()

latex_tab = tabulate(final_to_latex, headers='keys', tablefmt='latex_booktabs', showindex=False)
print(latex_tab)

\begin{tabular}{llrrrrr}
\toprule
 ('Model', '')   & ('Explanation Method', '')   &   ('COCO 20\^{}i', 'mIoULoss@0.01') &   ('COCO 20\^{}i', 'mIoULoss@0.05') &   ('Pascal 5\^{}i', 'IAUC@0.01') &   ('Pascal 5\^{}i', 'mIoULoss@0.01') &   ('Pascal 5\^{}i', 'mIoULoss@0.05') \\
\midrule
 DCAMA           & Random                       &                            33.79 &                            29.78 &                         44.44 &                             48.72 &                             34.64 \\
 DCAMA           & Gaussian Noise Mask          &                            27.07 &                            12.96 &                         46.95 &                             33.6  &                             16.01 \\
 DCAMA           & Saliency                     &                            33.93 &                            24.13 &                         39.69 &                             50.06 &                             32.8  \\
 DCAMA           & Integrated Gradients   